In [28]:
from langgraph.types import interrupt, Command
from langgraph.graph import StateGraph, MessagesState
from langgraph.checkpoint.memory import MemorySaver


def human_node(state: MessagesState):
    value = interrupt(f"what should i say in response to {state['messages']}")
    value_2 = interrupt(f"he should i say in response to {state['messages']}")
    return {'messages': [{'role': 'assistant', 'content': value}]}


checkpointer = MemorySaver()
graph_builder = StateGraph(MessagesState)
graph_builder.add_node(human_node)
graph_builder.set_entry_point("human_node")

graph = graph_builder.compile(checkpointer=checkpointer)


thread_config = {'configurable': {'thread_id': 'some_id'}}

graph.invoke({'messages': [{'role': 'user', 'content': 'hi'}]}, config=thread_config)

{'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='a7cb7662-20b3-4da9-9343-9b7b219b5d37')],
 '__interrupt__': [Interrupt(value="what should i say in response to [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='a7cb7662-20b3-4da9-9343-9b7b219b5d37')]", id='459d80a743481bcbcdb4373f4007fb11')]}

In [29]:
state = graph.get_state(thread_config)
state.tasks[0].interrupts

(Interrupt(value="what should i say in response to [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='a7cb7662-20b3-4da9-9343-9b7b219b5d37')]", id='459d80a743481bcbcdb4373f4007fb11'),)

In [27]:
state.tasks[0].interrupts

(Interrupt(value="what should i say in response to [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='ccd59aab-5ff1-47bf-a08d-b6e89e579e9f')]", id='5afac82e6e7b74559fe7040570210c3b'),)

In [16]:
state = graph.get_state(thread_config)
state.tasks[0].interrupts

(Interrupt(value="what should i say in response to [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='ccd59aab-5ff1-47bf-a08d-b6e89e579e9f')]", id='5afac82e6e7b74559fe7040570210c3b'),)

In [17]:
graph.invoke(Command(resume="hows it going"), config=thread_config)

{'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='ccd59aab-5ff1-47bf-a08d-b6e89e579e9f'),
  AIMessage(content='hows it going', additional_kwargs={}, response_metadata={}, id='60d73514-a328-44c5-8193-e0e45aff27e5', tool_calls=[], invalid_tool_calls=[])]}

In [18]:
from typing import Literal, Optional, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


class ApprovalState(TypedDict):
    action_details: str
    status: Optional[Literal["pending", "approved", "rejected"]]


def approval_node(state: ApprovalState) -> Command[Literal["proceed", "cancel"]]:
    # Expose details so the caller can render them in a UI
    decision = interrupt(
        {
            "question": "Approve this action?",
            "details": state["action_details"],
        }
    )

    # Route to the appropriate node after resume
    return Command(goto="proceed" if decision else "cancel")


def proceed_node(state: ApprovalState):
    return {"status": "approved"}


def cancel_node(state: ApprovalState):
    return {"status": "rejected"}


builder = StateGraph(ApprovalState)
builder.add_node("approval", approval_node)
builder.add_node("proceed", proceed_node)
builder.add_node("cancel", cancel_node)
builder.add_edge(START, "approval")
builder.add_edge("proceed", END)
builder.add_edge("cancel", END)

# Use a more durable checkpointer in production
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "approval-123"}}
initial = graph.stream_events(
    {"action_details": "Transfer $500", "status": "pending"},
    config=config,
    version="v3",
)
_ = initial.output  # drive the stream to completion
print(initial.interrupts)  # -> (Interrupt(value={'question': ..., 'details': ...}),)

# Resume with the decision; True routes to proceed, False to cancel
resumed = graph.stream_events(Command(resume=False), config=config, version="v3")
print(resumed.output["status"])

[Interrupt(value={'question': 'Approve this action?', 'details': 'Transfer $500'}, id='d7755645a0e9f5ad0603ae9945369cbe')]
rejected


In [25]:
from typing import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


class ReviewState(TypedDict):
    generated_text: str


def review_node(state: ReviewState):
    # Ask a reviewer to edit the generated content
    updated = interrupt(
        {
            "instruction": "Review and edit this content",
            "content": state["generated_text"],
        }
    )
    return {"generated_text": updated}


builder = StateGraph(ReviewState)
builder.add_node("review", review_node)
builder.add_edge(START, "review")
builder.add_edge("review", END)

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "review-42"}}
initial = graph.stream_events(
    {"generated_text": "Initial draft"}, config=config, version="v3"
)
_ = initial.output  # drive the stream to completion

hitl_answer = input(f"{initial.interrupts[0].value['instruction']}: {initial.interrupts[0].value['content']}")  # -> (Interrupt(value={'instruction': ..., 'content': ...}),)

# Resume with the edited text from the reviewer
final_state = graph.stream_events(
    Command(resume=hitl_answer),
    config=config,
    version="v3",
)
print(final_state.output["generated_text"])  # -> "Improved draft after review"

good!
